# Masked Self-Attention

Masked Self-Attention is a part of the **decoder architecture** in Transformers.

```text
The Transformer decoder is autoregressive at inference time and non-autoregressive at training time.
```

---

# Breaking Down the Sentence

## Inference

Inference means the prediction phase, where the trained model generates outputs for new inputs.

---

## Autoregressive Models

In deep learning, an autoregressive model generates outputs sequentially, where each new output depends on previously generated outputs.

Example:

In stock price prediction:

- Friday’s prediction may depend on:
  - Thursday’s price
  - Wednesday’s price
  - earlier values

So future values are generated using previous values.

---

# Autoregressive Nature of Language Models

Language models are autoregressive because the next word depends on previous words.

Example:

```text
I am going to ___
```

The next word prediction depends on the earlier words in the sequence.

So text generation happens sequentially.

---

# Transformer Decoder Behavior

The Transformer decoder is:

- **Autoregressive during inference**
  - words are generated one by one
  - each prediction depends on previously generated words

- **Non-autoregressive during training**
  - all words are processed in parallel

This parallel training is possible because of **Masked Self-Attention**.

# Example: Translation Model

Suppose we want to build a translation model, for example translating English to Nepali.

Example:

```text
I am fine  →  म ठिक छु
```

---

# Inference Phase

During inference (prediction):

1. The encoder converts each input word into vectors.
2. These vectors are sent to the decoder along with a `<START>` token.

---

The decoder now generates words sequentially, one after another, making the process **autoregressive**.

For example:

- Decoder receives `<START>`
- Predicts:

```text
म
```

This predicted word is again fed back into the decoder along with the encoder vectors.

Now the decoder predicts the next word.

Suppose it predicts:

```text
बेठीक
```

This output is again passed back into the decoder, and eventually the decoder predicts:

```text
छु
```

Finally, the decoder predicts:

```text
<END>
```

![](./images/img32.png)

---

# Training Phase

During training, instead of feeding the decoder’s previous prediction back into itself, we feed the correct word from the training data.

For example:

Instead of feeding:

```text
बेठीक
```

we feed the correct word:

```text
ठिक
```

and then compute the loss function.

This technique is called **Teacher Forcing**.

---

# Why Not Use Fully Autoregressive Training?

If training were fully autoregressive:

- words would need to be generated one by one
- training would become very slow

However, during training we already have the correct target sentence available.

So instead of waiting for previous predictions, we directly use the ground truth words.

This makes training **non-autoregressive** and allows parallel computation, greatly increasing training speed.

# Problem in Parallelizing the Decoder

The first block in the decoder is **Masked Multi-Head Attention**.

![](./images/img1.png)

Suppose we input the Nepali sentence:

```text
म ठिक छु
```

into the decoder embedding layer.

The Multi-Head Attention block converts these embeddings into contextual embeddings.

---

# Problem with Normal Self-Attention

In normal self-attention, every token can attend to all other tokens.

Since all words are processed in parallel, the current token can also see future tokens.

Example:

While predicting:

```text
म
```

the model could already see:

```text
ठिक छु
```

This creates **data leakage**, because during prediction the model should not know future words.

---

# Solution: Masked Self-Attention

To prevent this, we use a **mask matrix**.

The mask blocks attention to future tokens by setting their attention weights to zero (or \(-\infty\) before softmax).

So:

- a token can only attend to:
  - itself
  - previous tokens

but not future tokens.

---

# Example

When predicting:

```text
ठिक
```

the model can attend to:

```text
म
ठिक
```

but not:

```text
छु
```

---

# How the Mask Works

In the self-attention score matrix, future positions are masked.

This ensures that future token contributions become zero after softmax.

![](./images/img33.png)

---

# Final Idea

Masked Self-Attention allows the decoder to:

- train in parallel
- avoid future information leakage
- still behave autoregressively during prediction